# UltraMedical-Preference — Analyse Exploratoire (EDA)

## Synthèse & Observations

### Structure du dataset
- **Langue :** Anglais | **Source :** Tsinghua (`TsinghuaC3I/UltraMedical-Preference` sur HuggingFace)
- Format **préférentiel** (chosen/rejected) — conversations générées par IA et annotées par préférence
- 1 seul split : `train` (109 353 lignes) — plus grande source du corpus
- Colonnes : `prompt_id`, `label_type`, `prompt`, `chosen` (list[dict]), `rejected` (list[dict]), `metadata`, `feedback`
- `chosen` et `rejected` = listes de messages `{role: user|assistant, content: ...}`
- **Toujours exactement 2 messages** par conversation (1 user + 1 assistant) — vérifié

### Points d'attention
- Format conversationnel → nécessite une extraction avant usage en SFT
- `chosen` = réponse préférée (meilleure qualité) | `rejected` = réponse moins bonne
- `label_type` : critère de sélection des préférences (length, easy, hard...)
- `prompt_id` encode la source d'origine (ex: `MedMCQA,11404`, `ChatDoctor,33567`, `WikiInstruct,8304`)
- `metadata` et `feedback` : évaluations qualitatives narratives → non pertinentes pour le SFT
- **Double usage possible** : `chosen` → SFT | paires `chosen/rejected` → DPO (alignement par préférences)

### Décisions de nettoyage → `src/processing/ultramed_cleaning.py`
- Extraire le 1er message `user` de `chosen` → colonne `question`
- Extraire le 1er message `assistant` de `chosen` → colonne `answer`
- Supprimer les doublons
- Normaliser en minuscules | Ajouter `dataset_name = "ultramed"`
- **Conserver les paires `chosen/rejected` originales** pour l'entraînement DPO ultérieur

---
## 1. Chargement des données

In [ ]:
from datasets import load_from_disk
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
ultraMed_Dataset = load_from_disk("gs://p14-medical-data/raw_data/UltraMedical_dataset")
print(ultraMed_Dataset)

---
## 2. Exploration de la structure

In [ ]:
df = pd.DataFrame(ultraMed_Dataset["train"])
print(f"Shape : {df.shape}")
print(f"Colonnes : {list(df.columns)}")
df.head(3)

In [ ]:
df.isna().sum()

---
## 3. Analyse du format conversationnel

In [ ]:
# Nombre de messages par conversation (chosen)
df["nb_messages_chosen"] = df["chosen"].apply(len)
print("Distribution du nombre de messages dans 'chosen' :")
print(df["nb_messages_chosen"].value_counts())

In [ ]:
# Structure d'un message chosen : liste de dicts {role, content}
example = df["chosen"].iloc[0]
print(f"Nombre de messages : {len(example)}")
for msg in example:
    print(f"  role  : {msg['role']}")
    print(f"  content (extrait) : {msg['content'][:200]}")
    print()

In [ ]:
# Vérification que le pattern user/assistant est systématique
roles_chosen = df["chosen"].apply(lambda msgs: tuple(m["role"] for m in msgs))
print("Patterns de rôles dans 'chosen' :")
print(roles_chosen.value_counts())

---
## 4. Distribution des variables clés

In [ ]:
# Distribution du label_type (critère de sélection de la préférence)
df["label_type"].value_counts()

In [ ]:
# Sources d'origine des conversations (encodées dans prompt_id)
df["source"] = df["prompt_id"].str.split(",").str[0]
print("Top 15 sources d'origine :")
print(df["source"].value_counts().head(15))

In [ ]:
# Extraction des questions et réponses pour analyse des longueurs
df["question_text"] = df["chosen"].apply(
    lambda msgs: next((m["content"] for m in msgs if m["role"] == "user"), None)
)
df["answer_text"] = df["chosen"].apply(
    lambda msgs: next((m["content"] for m in msgs if m["role"] == "assistant"), None)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["question_text"].str.len(), bins=50, ax=axes[0])
axes[0].set_title("Distribution longueur des questions (en caractères)")
axes[0].set_xlabel("Nombre de caractères")

sns.histplot(df["answer_text"].str.len(), bins=50, ax=axes[1])
axes[1].set_title("Distribution longueur des réponses chosen (en caractères)")
axes[1].set_xlabel("Nombre de caractères")

plt.tight_layout()
plt.show()

---
## 5. Analyse des doublons

In [ ]:
# Doublons sur le prompt
print(f"Doublons sur 'prompt' : {df.duplicated(subset=['prompt']).sum()}")
print(f"Doublons sur 'prompt_id' : {df.duplicated(subset=['prompt_id']).sum()}")

---
## 6. Aperçu des paires chosen/rejected (usage DPO)

In [ ]:
# Exemple d'une paire chosen/rejected pour l'entraînement DPO
sample = df.iloc[2]
print("=== PROMPT ===")
print(sample["prompt"][:300])
print()
print("=== CHOSEN (réponse préférée) ===")
chosen_answer = next(m["content"] for m in sample["chosen"] if m["role"] == "assistant")
print(chosen_answer[:400])
print()
print("=== REJECTED (réponse moins bonne) ===")
rejected_answer = next(m["content"] for m in sample["rejected"] if m["role"] == "assistant")
print(rejected_answer[:400])